In [0]:
from pyspark.sql import functions as F

In [0]:
customer_sales = spark.sql("""
SELECT
    c.customer_id,
    c.customer_segment,
    COUNT(DISTINCT o.order_id) AS total_orders,
    SUM(o.total_amount) AS total_revenue,
    AVG(o.total_amount) AS average_order_value,
    MAX(o.order_date) AS last_order_date
FROM workspace.silver.customers_scd2 c
JOIN workspace.silver.orders o
    ON c.customer_id = o.customer_id
WHERE c.is_current = true
GROUP BY
    c.customer_id,
    c.customer_segment
""")

customer_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.customer_sales")

In [0]:
product_performance = spark.sql("""
SELECT
    p.product_id,
    p.product_name,
    p.category,

    COUNT(DISTINCT o.order_id) AS total_orders,

    SUM(o.quantity) AS units_sold,

    SUM(o.quantity * o.unit_price) AS revenue

FROM workspace.silver.products p

JOIN workspace.silver.order_items o
    ON p.product_id = o.product_id

GROUP BY
    p.product_id,
    p.product_name,
    p.category
""")

product_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.product_performance")

display(product_performance)

In [0]:
daily_sales = spark.sql("""
SELECT
    CAST(order_date AS DATE) AS order_date,
    COUNT(DISTINCT order_id) AS total_orders,
    SUM(total_amount) AS total_revenue,
    AVG(total_amount) AS average_order_value
FROM workspace.silver.orders
GROUP BY CAST(order_date AS DATE)
""")

daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.daily_sales_v2")

In [0]:
%sql
select *from workspace.gold.daily_sales_v2

In [0]:
category_sales = spark.sql("""
SELECT
    p.category,
    SUM(oi.quantity) AS units_sold,
    SUM(oi.quantity * oi.unit_price) AS revenue
FROM workspace.silver.order_items oi
JOIN workspace.silver.products p
    ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY revenue DESC
""")

category_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.category_sales_v2")

In [0]:
%sql
select *from workspace.gold.category_sales_v2

In [0]:
customer_activity = spark.sql("""
SELECT
    customer_id,
    COUNT(*) AS total_events,
    COUNT(DISTINCT session_id) AS total_sessions,

    SUM(
        CASE
            WHEN event_type = 'VIEW_PRODUCT' THEN 1
            ELSE 0
        END
    ) AS product_views,

    SUM(
        CASE
            WHEN event_type = 'ADD_TO_CART' THEN 1
            ELSE 0
        END
    ) AS cart_additions,

    SUM(
        CASE
            WHEN event_type = 'PURCHASE' THEN 1
            ELSE 0
        END
    ) AS purchases

FROM workspace.bronze.web_events

GROUP BY customer_id
""")

customer_activity.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.customer_activity")

display(customer_activity)

In [0]:
%sql
SELECT *
FROM workspace.gold.customer_activity
ORDER BY total_events DESC;